# HICO-DET: a 3D body fit for every person box

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/hico_det_bodies.ipynb)

HICO-DET (v20160224) draws a fresh box around a person for each action they take, so one person can hold several boxes. This notebook fits a 3D body with SAM 3D Body to every human box of 1,000 HICO-DET train photos (`pose.predict_many`: a photo's boxes in one pass). Boxes whose fits coincide in the picture are one person (`pose.apart`), and each fit gives that person's facing, head and torso in the picture's terms.

The box list comes from the private dataset `AbstractPhil/per-person-trial` (`inputs/hico_det_boxes.json`: each box with the HOI entries that list it, and each photo's sha256), and the fits ship back to its `fits/` folder. The photos come from the official HICO-DET archive ([umich-ywchao-hico.github.io](https://umich-ywchao-hico.github.io/)), streamed once from its Google Drive link: only the listed photos are kept on the runtime, each checked against its sha256, and no photo leaves the runtime. HICO-DET states no data licence: research use only. The SAM 3D Body weights are gated: request access to [facebook/sam-3d-body-vith](https://huggingface.co/facebook/sam-3d-body-vith) with the account of the `HF_TOKEN` secret first (the SAM License).

Runtime: a GPU (the G4, RTX PRO 6000) and the `HF_TOKEN` secret with read and write access to the private dataset. Run C1, restart the session (Runtime > Restart session), then C2 (the photos), C3 (the smoke: 20 photos, read before the rest) and C4 (every photo; it uses C3's fitter).

In [ ]:
# C1: install, then Runtime > Restart session
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.53")
pip("roma", "pytorch-lightning", "yacs", "omegaconf", "timm", "einops", "braceexpand")   # SAM 3D Body's inference path (pose.NEEDS)
print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: the listed photos from the official HICO-DET archive (one streamed pass; the archive itself is never stored)
import hashlib, json, os, tarfile, time
import requests
from huggingface_hub import hf_hub_download
from qwen_symbolic_thinking.hub import token

REPO = "AbstractPhil/per-person-trial"   # private
URL = "https://drive.usercontent.google.com/download?id=1dUByzVzM6z1Oq4gENa1-t0FLhr0UtDaS&export=download&confirm=t"
PHOTOS = "/content/hico_photos"
BOXES = json.load(open(hf_hub_download(REPO, "inputs/hico_det_boxes.json", repo_type="dataset", token=token()), encoding="utf-8"))
SHA = {p["file"]: p["sha256"] for p in BOXES["photos"]}


def good(name):
    path = os.path.join(PHOTOS, name)
    if not os.path.exists(path):
        return False
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest() == SHA[name]


class Archive:
    """The archive as a read-only stream; a dropped connection resumes at the same byte (an HTTP Range request)."""

    def __init__(self):
        self.pos, self.t0, self.shown, self.resumes = 0, time.time(), 0.0, 0
        self.resp = self._open()
        self.total = int(self.resp.headers.get("content-length") or 0) or None

    def _open(self):
        head = {"User-Agent": "Mozilla/5.0"}
        if self.pos:
            head["Range"] = f"bytes={self.pos}-"
        r = requests.get(URL, headers=head, timeout=120, stream=True)
        if r.status_code not in (200, 206) or r.headers.get("content-type", "").startswith("text/html"):
            raise SystemExit(f"Google Drive answered a page, not the archive ({r.status_code}); try later: {r.text[:300]}")
        if self.pos and r.status_code != 206:
            raise SystemExit("Google Drive refused to resume the stream; run C2 again (the photos kept so far stay)")
        return r

    def read(self, n=-1):
        n = n if n and n > 0 else 1 << 20
        while True:
            try:
                b = self.resp.raw.read(n)
            except Exception as e:   # a dropped connection: resume at the same byte
                b = None
                print("  read error:", type(e).__name__, str(e)[:120], flush=True)
            if b or (b == b"" and self.total and self.pos >= self.total):
                break
            self.resumes += 1
            if self.resumes > 5:
                raise SystemExit("the stream dropped six times; run C2 again (the photos kept so far stay)")
            time.sleep(5)
            self.resp = self._open()
        self.pos += len(b)
        if time.time() - self.shown > 30:
            spent = time.time() - self.t0
            rate = self.pos / spent if spent else 0
            left = (self.total - self.pos) / rate if self.total and rate else float("nan")
            print(f"  {self.pos / 1e9:.2f} of {(self.total or 0) / 1e9:.2f} GB, {spent / 60:.1f} min spent, at most "
                  f"{left / 60:.1f} min left ({rate / 1e6:.0f} MB/s)", flush=True)
            self.shown = time.time()
        return b


os.makedirs(PHOTOS, exist_ok=True)
need = {f for f in SHA if not good(f)}
print(f"{len(SHA)} photos listed, {len(SHA) - len(need)} already here", flush=True)
t0, bad = time.time(), []
if need:
    arc = Archive()
    with tarfile.open(fileobj=arc, mode="r|gz", bufsize=1 << 20) as tar:
        for m in tar:
            path = m.name.replace("\\", "/")
            name = path.rsplit("/", 1)[-1]
            if not (m.isfile() and "/train2015/" in path and name in need):
                continue
            data = tar.extractfile(m).read()
            if hashlib.sha256(data).hexdigest() != SHA[name]:
                bad.append(name)
                continue
            with open(os.path.join(PHOTOS, name), "wb") as fh:
                fh.write(data)
            need.discard(name)
            if not need:
                break   # the train photos come before the test photos in the archive: the rest is never read
    arc.resp.close()
print(f"photos here: {len(SHA) - len(need)} of {len(SHA)}; not found {len(need)}; checksum differs {len(bad)} {bad[:3]}; "
      f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
# C3: the fitter (every box of a photo in one pass), its rows shipped to fits/<stamp>_<tag>/; the smoke: the first 20 photos
import json, os, statistics, time
import torch
from PIL import Image
import qwen_symbolic_thinking
from qwen_symbolic_thinking import pose
from qwen_symbolic_thinking.hub import ship

FIT = pose.load()


def inside(rec):
    """The share of the fit's body points inside its own box: high when the keypoints are in the picture's pixels."""
    x0, y0, x1, y1 = rec["box"]
    pts = [rec["keypoints_2d"][i] for i in pose.BODY_2D]
    return sum(x0 <= x <= x1 and y0 <= y <= y1 for x, y in pts) / len(pts)


def run(photos, tag):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    out = f"/content/fits/{stamp}_{tag}"
    os.makedirs(out, exist_ok=True)
    n_boxes = sum(len(p["boxes"]) for p in photos)
    print(f"{len(photos)} photos, {n_boxes} boxes", flush=True)
    t0, shown, done, fitted, ins, gaps = time.time(), time.time(), 0, 0, [], []
    with open(f"{out}/rows.jsonl", "w", encoding="utf-8") as fh:
        for i, p in enumerate(photos, 1):
            with Image.open(os.path.join(PHOTOS, p["file"])) as im:
                recs = pose.predict_many(FIT, im, p["boxes"])
            fh.write(json.dumps({"file": p["file"], "width": p["width"], "height": p["height"], "fits": recs}) + "\n")
            done += len(recs)
            got = [r for r in recs if r]
            fitted += len(got)
            ins += [inside(r) for r in got]
            gaps += [pose.apart(a, b) for j, a in enumerate(got) for b in got[j + 1:]]
            if time.time() - shown > 60 or i == len(photos):
                spent = time.time() - t0
                print(f"  {i} of {len(photos)} photos ({done} of {n_boxes} boxes), {spent / 60:.1f} min spent, "
                      f"~{spent / done * (n_boxes - done) / 60:.1f} min left", flush=True)
                shown = time.time()
    spent = time.time() - t0
    gaps = sorted(g for g in gaps if g is not None)
    summary = {"stamp": stamp, "tag": tag, "photos": len(photos), "boxes": n_boxes, "fitted": fitted, "no_body": n_boxes - fitted,
               "minutes": round(spent / 60, 2), "s_per_photo": round(spent / max(len(photos), 1), 4),
               "s_per_box": round(spent / max(n_boxes, 1), 4), "gpu": torch.cuda.get_device_name(0),
               "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1), "weights": FIT["weights"],
               "sam_3d_body_commit": FIT["commit"], "library": qwen_symbolic_thinking.__version__,
               "inside_own_box_median": round(statistics.median(ins), 3) if ins else None, "pairs_in_a_photo": len(gaps),
               "apart_quantiles": {str(q): gaps[int(q * (len(gaps) - 1))] for q in (0.05, 0.1, 0.25, 0.5)} if gaps else None,
               "apart_under": {str(t): sum(g < t for g in gaps) for t in (0.02, 0.05, 0.1, 0.2)}}
    with open(f"{out}/summary.json", "w", encoding="utf-8") as fh:
        json.dump(summary, fh, indent=1)
    url = ship(out, f"fits/{stamp}_{tag}", f"fits {stamp}_{tag}: {fitted} of {n_boxes} boxes on {len(photos)} photos",
               repo=REPO, repo_type="dataset")
    print("shipped:", url)
    return summary


summary = run(BOXES["photos"][:20], "hico_det_bodies_smoke20")
print(json.dumps(summary, indent=1))

In [ ]:
# C4: every photo, after C3's smoke reads right (the fits inside their own boxes; a gap between one person's boxes and two people)
summary = run(BOXES["photos"], "hico_det_bodies")
print(json.dumps(summary, indent=1))